# Adquisición y Cruce de Datos
**Proyecto:** Predicción de Satisfacción en Aerolíneas según Servicio y Perfil Sociodemográfico.
**Objetivo de este notebook:** Leer las fuentes originales, realizar el cruce estadístico por edad y género, y consolidar el diccionario de datos.

In [14]:
import pandas as pd
import numpy as np

# Configuración para ver todas las columnas si el dataset es muy ancho
pd.set_option('display.max_columns', None)

Se cargan los datasets desde la carpeta `/data`. 
* **Fuente 1:** Pasajeros de Aerolínea (Kaggle)
* **Fuente 2:** Censo de Ingresos Adultos (Kaggle)

In [15]:
# Leer Fuente 1: Pasajeros
pasajeros = pd.read_csv('../data/airline_passenger_satisfaction.csv')
print(f"Dimensiones de Pasajeros: {pasajeros.shape}")
display(pasajeros.head(3))

# Leer Fuente 2: Censo
censo = pd.read_csv('../data/adult_census_income.csv')
print(f"\nDimensiones de Censo: {censo.shape}")
display(censo.head(3))

Dimensiones de Pasajeros: (103904, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,satisfied



Dimensiones de Censo: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K


Los nombres de las columnas varían entre los datasets (ej. `Age` vs `age`, `Gender` vs `sex`). se van a estandarizar en el censo para que coincidan con los de pasajeros. 
Además, se creara una variable numérica `ingreso_alto` en el censo para poder calcular promedios.

In [16]:
# Estandarizar nombres de columnas en el Censo
censo = censo.rename(columns={'age': 'Age', 'sex': 'Gender'})

# El censo americano usa "Male" y "Female", pero a veces con espacios extra. 
# Se asegura el mismo formato de la fuente de pasajeros.
censo['Gender'] = censo['Gender'].str.strip().str.capitalize()
pasajeros['Gender'] = pasajeros['Gender'].str.strip().str.capitalize()

# 2.2 Convertir el ingreso del censo a numérico (1 si gana >50K, 0 si no)
# Ajusta '>50K' según cómo venga escrito en el CSV (a veces trae un punto final '>50K.')
censo['ingreso_alto'] = np.where(censo['income'].str.contains('>50K'), 1, 0)

**Caso de cruce por aproximación:** Dado que no se tienen a las mismas personas físicas, calcularemos un estimado del poder adquisitivo (probabilidad de ingresos altos) agrupando por **Edad y Género** en el censo, y lo inyectaremos a los pasajeros.

In [17]:
# Calcular el promedio estadístico de ingresos por perfil
ingreso_promedio = censo.groupby(["Age", "Gender"])["ingreso_alto"].mean().reset_index()
ingreso_promedio = ingreso_promedio.rename(columns={'ingreso_alto': 'probabilidad_altos_ingresos'})

# Verificación antes del cruce
filas_antes = pasajeros.shape[0]
print(f"Filas en dataset principal ANTES del cruce: {filas_antes}")

#se hace cruce
pasajeros_cruzado = pd.merge(pasajeros, ingreso_promedio, on=["Age", "Gender"], how="left")

# Verificación después del cruce
filas_despues = pasajeros_cruzado.shape[0]
print(f"Filas en dataset principal DESPUÉS del cruce: {filas_despues}")

# Comprobación de integridad
if filas_antes == filas_despues:
    print("¡Éxito! El cruce no duplicó ni perdió pasajeros.")
else:
    print("Advertencia: Las filas cambiaron. Revisar duplicados en las llaves.")

display(pasajeros_cruzado[['id', 'Age', 'Gender', 'probabilidad_altos_ingresos', 'satisfaction']].head())

Filas en dataset principal ANTES del cruce: 103904
Filas en dataset principal DESPUÉS del cruce: 103904
¡Éxito! El cruce no duplicó ni perdió pasajeros.


,id,Age,Gender,probabilidad_altos_ingresos,satisfaction
0,70172,13,Male,NaN,neutral or dissatisfied
1,5047,25,Male,0.071970,neutral or dissatisfied
2,110028,26,Female,0.048276,satisfied
3,24026,25,Female,0.047923,neutral or dissatisfied
4,119299,61,Male,0.407240,satisfied


A continuación, el diccionario de las columnas principales del dataset ya consolidado (`pasajeros_cruzado`), el cual responde directamente a nuestro objetivo de negocio.

| Columna | Tipo de Dato (Pandas) | Significado | Fuente Origen |
| :--- | :--- | :--- | :--- |
| `id` | int64 | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers |
| `Gender` | object | Género del pasajero (Male / Female). | Kaggle: Airline Passengers |
| `Age` | int64 | Edad real del pasajero en años. | Kaggle: Airline Passengers |
| `Customer Type` | object | Nivel de lealtad (Loyal customer / disloyal customer). | Kaggle: Airline Passengers |
| `Type of Travel` | object | Motivo del viaje (Personal Travel / Business Travel). | Kaggle: Airline Passengers |
| `Class` | object | Cabina en la que viajó (Business, Eco, Eco Plus). | Kaggle: Airline Passengers |
| `Inflight wifi service` | int64 | Calificación del WiFi (0 a 5). | Kaggle: Airline Passengers |
| `Leg room service` | int64 | Calificación del espacio para piernas (0 a 5). | Kaggle: Airline Passengers |
| `probabilidad_altos_ingresos` | float64 | Valor inferido (0.0 a 1.0). Indica la probabilidad de que el pasajero gane +50K al año basado en su edad y género. | Kaggle: Adult Census |
| `Satisfaction` | object | Variable Objetivo. Si quedó satisfecho o insatisfecho. | Kaggle: Airline Passengers |

# 5. Calidad de Datos

## 5.1. Autodiagnóstico post-cruce
Antes de limpiar las variables individuales, verificamos tres criterios de integridad sobre `pasajeros_cruzado`:
1. Que el número de filas se haya mantenido intacto respecto a la fuente principal.
2. Que no existan identificadores (`id`) duplicados.
3. Cuántos valores nulos existen en el dataset después del cruce.

In [18]:
# 1. Confirmación de filas
print(f"Filas antes del cruce: {pasajeros.shape[0]} | Filas después del cruce: {pasajeros_cruzado.shape[0]}")

# 2. Revisión de IDs duplicados
duplicados_id = pasajeros_cruzado['id'].duplicated().sum()
print(f"IDs duplicados después del cruce: {duplicados_id}")

# 3. Revisión de columnas con valores nulos
nulos = pasajeros_cruzado.isna().sum()
print("\nColumnas con valores nulos detectados:")
print(nulos[nulos > 0])

Filas antes del cruce: 103904 | Filas después del cruce: 103904
IDs duplicados después del cruce: 0

Columnas con valores nulos detectados:
Arrival Delay in Minutes        310
probabilidad_altos_ingresos    6947
dtype: int64


## 5.2. Revisión y corrección de tipos de dato (`dtypes`)

Inspeccionamos los tipos de dato actuales de `pasajeros_cruzado` y realizamos tres correcciones técnicas:
1. **Columna `Unnamed: 0` (si existe):** Es un índice residual del archivo CSV original que no aporta información; se elimina.
2. **Identificador `id` (`int64` -> `object`/`str`):** Aunque contiene dígitos, un ID no es una magnitud matemática (promediar o sumar IDs sería un error). Lo convertimos a texto.
3. **Retraso de llegada `Arrival Delay in Minutes` (`float64` -> `Int64`) y variables categóricas (`Class`):** `Arrival Delay in Minutes` aparece como decimal (`float64`) únicamente por tener valores vacíos, mientras que `Departure Delay in Minutes` es entero (`int64`). Lo pasamos a entero nulo (`Int64`). Además, definimos `Class` como categoría ordenada (`Eco` < `Eco Plus` < `Business`) para respetar su jerarquía real.

In [19]:
print("--- Tipos de dato ANTES de corregir ---")
print(pasajeros_cruzado.dtypes)

# 1. Eliminar columna índice residual del CSV si está presente
if 'Unnamed: 0' in pasajeros_cruzado.columns:
    pasajeros_cruzado = pasajeros_cruzado.drop(columns=['Unnamed: 0'])

# 2. Convertir 'id' a texto (string/object) para evitar cálculos numéricos accidentales
pasajeros_cruzado['id'] = pasajeros_cruzado['id'].astype(str)

# 3. Convertir retraso de llegada a entero que soporta nulos (Int64)
if 'Arrival Delay in Minutes' in pasajeros_cruzado.columns:
    pasajeros_cruzado['Arrival Delay in Minutes'] = (
        pd.to_numeric(pasajeros_cruzado['Arrival Delay in Minutes'], errors='coerce')
        .round()
        .astype('Int64')
    )

# 4. Convertir 'Class' en categoría ordenada y variables nominales cerradas a 'category'
orden_clases = pd.CategoricalDtype(categories=['Eco', 'Eco Plus', 'Business'], ordered=True)
pasajeros_cruzado['Class'] = pasajeros_cruzado['Class'].astype(orden_clases)

for col_cat in ['Gender', 'Type of Travel', 'satisfaction']:
    pasajeros_cruzado[col_cat] = pasajeros_cruzado[col_cat].astype('category')

print("\n--- Tipos de dato DESPUÉS de corregir ---")
print(pasajeros_cruzado[['id', 'Class', 'Gender', 'Type of Travel', 'Arrival Delay in Minutes', 'satisfaction']].dtypes)

--- Tipos de dato ANTES de corregir ---
Unnamed: 0                             int64
id                                     int64
Gender                                object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Inflight service                       int64
Cleanliness    

## 5.3. Identificación y tratamiento de valores atípicos e inconsistencias

Exploramos las variables clave para nuestra pregunta de negocio (calificaciones de servicio, tipo de cliente y perfil demográfico) mediante `.describe()` y `.value_counts()`.

In [20]:
# Exploración numérica de variables clave del objetivo
print("--- Resumen estadístico de variables numéricas clave ---")
display(pasajeros_cruzado[['Age', 'Inflight wifi service', 'Leg room service', 'probabilidad_altos_ingresos']].describe())

# Exploración categórica de tipo de cliente
print("\n--- Frecuencia de categorías en Customer Type (ANTES) ---")
print(pasajeros_cruzado['Customer Type'].value_counts(dropna=False))

--- Resumen estadístico de variables numéricas clave ---


,Age,Inflight wifi service,Leg room service,probabilidad_altos_ingresos
count,103904.000000,103904.000000,103904.000000,96957.000000
mean,39.379706,2.729683,3.351055,0.224213
std,15.114964,1.327829,1.315605,0.159292
min,7.000000,0.000000,0.000000,0.000000
25%,27.000000,2.000000,2.000000,0.106227
50%,40.000000,3.000000,4.000000,0.186441
75%,51.000000,4.000000,4.000000,0.394277
max,85.000000,5.000000,5.000000,0.517937



--- Frecuencia de categorías en Customer Type (ANTES) ---
Customer Type
Loyal Customer       84923
disloyal Customer    18981
Name: count, dtype: int64


### Decisiones y justificación de limpieza

A partir de la exploración anterior, identificamos y tratamos **3 problemas reales de calidad de datos**:

1. **Inconsistencia en escala de calificaciones (`0` en escalas de 1 a 5 en `Inflight wifi service` y `Leg room service`):**
   * **Hallazgo:** El mínimo reportado es `0`, pero la encuesta califica servicios del **1 al 5**. En la documentación de la fuente, el valor `0` significa *"No aplica / Servicio no utilizado"* (no significa pésimo servicio).
   * **Decisión y justificación:** **Convertir los `0` en `NaN` (`np.nan`)** en las columnas de calificación de servicio. Si dejamos el `0` como número, reduciríamos artificialmente el promedio de satisfacción del WiFi y confundiríamos al futuro modelo predictivo haciéndole creer que el servicio falló cuando en realidad no aplicaba.

2. **Inconsistencia de escritura en `Customer Type` (`"Loyal Customer"` vs `"disloyal Customer"`):**
   * **Hallazgo:** La categoría tiene un formato de mayúsculas inconsistente (`Loyal` inicia en mayúscula y `disloyal` en minúscula).
   * **Decisión y justificación:** **Corregir y estandarizar** usando `.str.strip().str.lower().str.title()` para unificar el formato a `"Loyal Customer"` y `"Disloyal Customer"`, convirtiéndola luego a tipo `category` para evitar errores en gráficos y agrupaciones.

3. **Valores atípicos demográficos para el negocio y nulos del cruce (`Age < 17`, pasajeros desde los 7 años):**
   * **Hallazgo:** `Age` tiene un mínimo de **7 años**. Esto genera que todos los pasajeros entre 7 y 16 años tengan `NaN` en `probabilidad_altos_ingresos` (ya que el Censo de Adultos inicia a los 17 años). Además, un niño de 7 años no toma decisiones de presupuesto ni exige WiFi por productividad laboral.
   * **Decisión y justificación:** **No eliminar las filas, sino imputar y marcar**. Asignamos `0.0` en `probabilidad_altos_ingresos` a los menores de 17 años (pues legal y censalmente no perciben ingresos altos propios >50K USD) y creamos la columna bandera `es_menor_edad` (`True` si `Age < 18`) para poder filtrarlos o analizarlos por separado en la fase de modelado sin perder trazabilidad del dataset original.

In [21]:
total_filas = len(pasajeros_cruzado)

# --- TRATAMIENTO 1: Calificaciones en 0 ("No aplica") -> NaN ---
columnas_calificacion = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness'
]
# Filtramos por seguridad solo las que existan en el dataframe
columnas_calificacion = [c for c in columnas_calificacion if c in pasajeros_cruzado.columns]

ceros_wifi = (pasajeros_cruzado['Inflight wifi service'] == 0).sum()
ceros_legroom = (pasajeros_cruzado['Leg room service'] == 0).sum()

pasajeros_cruzado[columnas_calificacion] = pasajeros_cruzado[columnas_calificacion].replace(0, np.nan)

print(f"1. Calificaciones '0' convertidas a NaN en WiFi: {ceros_wifi} ({ceros_wifi/total_filas*100:.2f}% del total)")
print(f"   Calificaciones '0' convertidas a NaN en Leg room: {ceros_legroom} ({ceros_legroom/total_filas*100:.2f}% del total)")

# --- TRATAMIENTO 2: Estandarización de texto en 'Customer Type' ---
desleales_mal_escritos = (pasajeros_cruzado['Customer Type'].astype(str) == 'disloyal Customer').sum()
pasajeros_cruzado['Customer Type'] = (
    pasajeros_cruzado['Customer Type']
    .astype(str).str.strip().str.lower().str.title()
    .astype('category')
)
print(f"\n2. Registros estandarizados en 'Customer Type': {desleales_mal_escritos} ({desleales_mal_escritos/total_filas*100:.2f}% del total)")
print("   Categorías únicas DESPUÉS de limpiar:")
print(pasajeros_cruzado['Customer Type'].value_counts())

# --- TRATAMIENTO 3: Menores de edad y nulos en 'probabilidad_altos_ingresos' ---
nulos_menores = pasajeros_cruzado['probabilidad_altos_ingresos'].isna().sum()
pasajeros_cruzado['es_menor_edad'] = pasajeros_cruzado['Age'] < 18

# Asignamos 0.0 a menores de 17 años (fuera del rango del censo adulto)
pasajeros_cruzado.loc[pasajeros_cruzado['Age'] < 17, 'probabilidad_altos_ingresos'] = 0.0

# Si quedara algún nulo aislado en edades adultas extremas, se imputa con la mediana de su género
pasajeros_cruzado['probabilidad_altos_ingresos'] = (
    pasajeros_cruzado.groupby('Gender', observed=False)['probabilidad_altos_ingresos']
    .transform(lambda x: x.fillna(x.median()))
)

print(f"\n3. Nulos tratados en 'probabilidad_altos_ingresos' (menores <17 años): {nulos_menores} ({nulos_menores/total_filas*100:.2f}% del total)")
print(f"   Nulos restantes en 'probabilidad_altos_ingresos': {pasajeros_cruzado['probabilidad_altos_ingresos'].isna().sum()}")

1. Calificaciones '0' convertidas a NaN en WiFi: 3103 (2.99% del total)
   Calificaciones '0' convertidas a NaN en Leg room: 472 (0.45% del total)

2. Registros estandarizados en 'Customer Type': 18981 (18.27% del total)
   Categorías únicas DESPUÉS de limpiar:
Customer Type
Loyal Customer       84923
Disloyal Customer    18981
Name: count, dtype: int64

3. Nulos tratados en 'probabilidad_altos_ingresos' (menores <17 años): 6947 (6.69% del total)
   Nulos restantes en 'probabilidad_altos_ingresos': 0


## 5.4. Diccionario de Datos Actualizado (con Estado de Calidad)

| Columna | Tipo Final | Significado | Fuente Origen | Estado |
| :--- | :--- | :--- | :--- | :--- |
| `id` | object (str) | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers | Corregida (de int64 a str) |
| `Gender` | category | Género del pasajero (Male / Female). | Kaggle: Airline Passengers | Corregida (tipo category) |
| `Age` | int64 | Edad del pasajero en años (7 a 85). | Kaggle: Airline Passengers | Sin problemas (menores marcados) |
| `es_menor_edad` | bool | Bandera creada (`True` si `Age < 18`) para separar menores de adultos decisores. | Variable derivada | Nueva / Creada en limpieza |
| `Customer Type` | category | Nivel de lealtad (`Loyal Customer` / `Disloyal Customer`). | Kaggle: Airline Passengers | Corregida (mayúsculas unificadas) |
| `Type of Travel` | category | Motivo del viaje (`Personal Travel` / `Business Travel`). | Kaggle: Airline Passengers | Corregida (tipo category) |
| `Class` | category (ordenada) | Cabina (`Eco` < `Eco Plus` < `Business`). | Kaggle: Airline Passengers | Corregida (jerarquía ordinal) |
| `Inflight wifi service` | float64 | Calificación real del WiFi (1 a 5; `0` pasado a `NaN` por no aplicar). | Kaggle: Airline Passengers | Corregida (ceros tratados) |
| `Leg room service` | float64 | Calificación real de espacio para piernas (1 a 5; `0` pasado a `NaN`). | Kaggle: Airline Passengers | Corregida (ceros tratados) |
| `Arrival Delay in Minutes` | Int64 | Minutos de retraso al aterrizar (~310 nulos originales conservados). | Kaggle: Airline Passengers | Pendiente de revisar (imputación en modelado) |
| `probabilidad_altos_ingresos` | float64 | Estimación de ganar >50K USD anuales (0.0 asignado a menores de 17 años). | Kaggle: Adult Census | Corregida (nulos post-cruce tratados) |
| `satisfaction` | category | Variable objetivo (`neutral or dissatisfied` / `satisfied`). | Kaggle: Airline Passengers | Sin problemas (tipo category) |